In [86]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv('../data/temp/chats_edu.csv')

# quero que fique apenas as colunas relevantes
df = df[['session_id','timestamp','ESCUELA']]

# Remover linhas com valores nulos
df = df.dropna()

# Calcular contagem de mensagens por sessão
df['message_count'] = df.groupby('session_id')['session_id'].transform('count')

# Quero que olhe o timestamp e classifique como manhã (M), tarde (T) ou noite (N)
def classify_time_of_day(timestamp):
    hour = pd.to_datetime(timestamp).hour
    if 6 <= hour < 12:
        return 'M'  # Manhã
    elif 12 <= hour < 18:
        return 'T'  # Tarde
    else:
        return 'N'  # Noite
df['time_of_day'] = df['timestamp'].apply(classify_time_of_day)

df.head(10)


# Transformar as colunas categóricas em formato binário (one-hot encoding)
df_encoded = pd.get_dummies(df.drop(columns=['session_id', 'message_count']))

# Remover colunas numéricas (como NF) para garantir apenas valores booleanos/0/1
# df_encoded = df_encoded.drop(columns=['NF'])

# 1. Encontrar itemsets frequentes
frequent_itemsets = apriori(df_encoded, min_support=0.1, use_colnames=True)

# 2. Gerar regras com base nos itemsets
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

# 3. Ver as colunas principais
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])





                                      antecedents      consequents   support  \
0  (ESCUELA_ESCUELA DE ADMINISTRACIÓN Y NEGOCIOS)  (time_of_day_N)  0.106136   
1                 (ESCUELA_ESCUELA DE TECNOLOGÍA)  (time_of_day_N)  0.230373   

   confidence      lift  
0    0.787836  1.069745  
1    0.712854  0.967932  


In [87]:
# Formatar as regras para melhor visualização
print("As 10 regras de associação mais relevantes:\n")
for idx, row in rules_sorted.head(10).iterrows():
    antecedents = ', '.join(list(row['antecedents']))
    consequents = ', '.join(list(row['consequents']))
    print(f"Regra {idx + 1}:")
    print(f"SE {antecedents}")
    print(f"ENTÃO {consequents}")
    print(f"Support: {row['support']:.3f}")
    print(f"Confidence: {row['confidence']:.3f}")
    print(f"Lift: {row['lift']:.3f}")
    print("-" * 50)

As 10 regras de associação mais relevantes:

Regra 1:
SE ESCUELA_ESCUELA DE ADMINISTRACIÓN Y NEGOCIOS
ENTÃO time_of_day_N
Support: 0.106
Confidence: 0.788
Lift: 1.070
--------------------------------------------------
Regra 2:
SE ESCUELA_ESCUELA DE TECNOLOGÍA
ENTÃO time_of_day_N
Support: 0.230
Confidence: 0.713
Lift: 0.968
--------------------------------------------------
